# Datathon 2026 Playground — Student Performance Classification
## A CRISP-DM Walkthrough: the ordinal ranker, and an honest CV-vs-LB post-mortem

**Task.** Predict each student's performance level among 4 **ordered** classes
(`0` Very Low → `3` Excellent) from academic + behavioural tabular records.
**Metric.** Plain multi-class **accuracy** on the 800-row test set.

**What this notebook establishes.** Reframing the problem as **ordinal
decomposition** — estimating a latent academic *rank* from the cumulative
probabilities `Σₖ P(y > k)` and quartile-binning it — produces a genuinely strong,
independent ranker at **CV 0.5276 ± 0.0030** (seeds 42–46). It cleanly beats the
*untuned* reg/clf pair (0.5158) and is a valuable, diverse view of the target.

**Honest headline, after leaderboard feedback.** The ordinal model is **not** the
overall best. When the previous production model's **Optuna-tuned** reg/clf core is
scored in *this same harness* it reaches **0.5334** — above the ordinal's 0.5276 —
and the public LB confirmed it (**v3 0.57916 vs ordinal 0.57500**, a ~1–2 row gap).
My first draft compared ordinal to the *untuned* baseline and wrongly called it the
winner; corrected below.

| Model — identical 52-feature CV harness, seeds 42–46 | CV accuracy | std |
|---|---|---|
| XGB regressor only | 0.5117 | 0.0041 |
| reg + clf pair (**untuned**) | 0.5158 | 0.0051 |
| Ordinal decomposition Σ P(y>k) (tuned) | 0.5276 | 0.0030 |
| **v3 reg/clf core (Optuna-tuned)** — current best, LB 0.57916 | **0.5334** | 0.0021 |
| ordinal + v3-core rank-blend (diversification) | 0.5340–0.5343 | 0.0032 |

**Takeaway.** The ordinal ranker's real value is **diversity**, not raw CV. Blending
its rank with v3's tuned core nudges CV to ~0.534 — but that is **inside the noise
band** (±0.003) and does *not* clear the project's +0.002 promotion bar. It is a
hedge for private-score robustness, not a confident win.

> **Guardrail.** This notebook performs **no Kaggle submission**. It only trains,
> cross-validates, and writes candidate CSVs locally.


## CRISP-DM 1 — Business Understanding

Schools want to flag students needing support early. We must rank/classify each
student into one of four **ordered** performance bands.

Two facts shape every modelling decision:

1. **The labels are ordered**, not nominal. A model that respects the ordering
   (`0 < 1 < 2 < 3`) should out-rank one that treats the classes as unrelated.
2. **The metric is raw accuracy**, and — as we verify below — the classes are
   **balanced** (~800 each in train). Under a balanced target, the optimal way
   to turn a monotone latent *score* into labels is to cut it at **quartiles**.

Together these motivate the **latent-score → quartile** framing that the whole
project uses: estimate one continuous "academic rank" per student, then split it
25/25/25/25. Accuracy then depends **only on how well that score ranks students**
— so the entire game is *rank quality*.


In [1]:
import warnings, numpy as np, pandas as pd
from pathlib import Path
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier, XGBRegressor
warnings.filterwarnings("ignore")
pd.set_option("display.width", 120)

DATA = Path("kaggle")
train = pd.read_csv(DATA / "train.csv")
test  = pd.read_csv(DATA / "test.csv")
sample = pd.read_csv(DATA / "sample_submission.csv")
y = train["target"]
print("train", train.shape, "| test", test.shape)
print("target counts:", y.value_counts().sort_index().to_dict())


train (3200, 43) | test (800, 42)
target counts: {0: 813, 1: 796, 2: 784, 3: 807}


## CRISP-DM 2 — Data Understanding

### 2.1 Class balance → quartile binning is justified
The four classes are near-perfectly balanced (813 / 796 / 784 / 807). The test
set is assumed balanced too, so **quartile-binning a latent score forces exactly
200/200/200/200 predictions** — the Bayes-optimal decision for a monotone score
under a balanced prior.


In [2]:
feat_cols = [c for c in train.columns if c not in ("id", "target")]
raw_corr = (train[feat_cols]
            .apply(lambda col: np.corrcoef(col, y)[0, 1])
            .sort_values(key=lambda s: s.abs(), ascending=False))
print("Top raw |correlation| with target:")
print(raw_corr.head(8).round(3))
print("\nkelas: nunique =", train["kelas"].nunique(), "of", len(train), "rows",
      "-> near-unique identifier (useless as a category)")


Top raw |correlation| with target:
tugas_selesai        0.215
skor_tryout          0.114
aktivitas_hari_12   -0.045
aktivitas_hari_14    0.032
tugas_diberikan     -0.032
aktivitas_hari_06   -0.031
aktivitas_hari_08    0.027
aktivitas_hari_11   -0.027
dtype: float64

kelas: nunique = 786 of 3200 rows -> near-unique identifier (useless as a category)


### 2.2 The signal is *multivariate*, not univariate

Raw per-column correlations are tiny — the strongest single feature
(`tugas_selesai`) is only 0.215, and the 16 daily-activity columns and most
behavioural scores look like pure noise (|corr| < 0.05).

But **univariate correlation is misleading here.** When we group features into
blocks and measure their *joint* contribution to CV accuracy, the "noise" blocks
turn out to carry large signal:

| Feature block added to the signal core | CV accuracy | Δ |
|---|---|---|
| signal core (completion + weekly volatility) | 0.4354 | — |
| **+ daily-activity block** (16 "noise" cols) | 0.4889 | **+0.053** |
| **+ behavioural-scores block** (motivation, discipline, …) | 0.4812 | **+0.046** |
| + `kelas` encodings | 0.4345 | −0.001 |

**Takeaway that drives feature design:** do **not** feature-select on univariate
correlation. Keep every informative column and let a strong model exploit the
multivariate interactions. Only `kelas` (a near-unique ID) is genuinely dropped.


In [3]:
# Engineered signal axes: completion + weekly-grade volatility (the ordered driver)
W = [f"nilai_minggu_{i:02d}" for i in range(1, 13)]
Wm = train[W].to_numpy(float)
cr = (train["tugas_selesai"] / train["tugas_diberikan"].replace(0, np.nan)).fillna(0)
sig = pd.DataFrame({
    "compl_ratio": cr,
    "minggu_std": Wm.std(1),
    "abs_change_sum": np.abs(Wm).sum(1),
    "minggu_range": Wm.max(1) - Wm.min(1),
    "cum_std": np.cumsum(Wm, 1).std(1),
})
print("Engineered signal-core correlations with target:")
print(sig.apply(lambda col: np.corrcoef(col, y)[0, 1]).round(3).sort_values(ascending=False))


Engineered signal-core correlations with target:
compl_ratio       0.405
abs_change_sum    0.397
minggu_std        0.396
minggu_range      0.375
cum_std           0.361
dtype: float64


### 2.3 Train/test distribution shift — none

Before trusting CV, we confirm the engineered features are **stable** between
train and test (Population Stability Index). Max PSI across all 52 features is
**0.03** — nothing exceeds the 0.1 "mild shift" line — so local CV is a
trustworthy proxy and no feature needs to be dropped for instability.


## CRISP-DM 3 — Data Preparation

The feature spine (`build_features`) turns each student's raw record into 52
leakage-safe features:

* **Assignment behaviour** — completion ratio / diff / remaining.
* **Weekly grade *changes*** (`nilai_minggu_*` are deltas) → mean/std/range/IQR,
  slope, positive/negative step counts, late-minus-early, and the **cumulative
  trajectory** (`cumsum`) which reconstructs the student's actual grade *level*.
* **Daily-activity sequence** (16 days) → the same robust summary statistics.
* **Behavioural / background scores** — kept as-is (multivariate signal).
* **Signal interactions** — volatility × completion, tryout × completion, etc.

Everything is **target-free and frame-local** (no leakage). `kelas` is dropped.
The identical function is applied to test.


In [4]:
import sys
sys.path.insert(0, "scripts")
from cv_harness import build_features, quartile_bin, z, ev, SEEDS

def feats(df):
    f = build_features(df, "full")
    return f.drop(columns=[col for col in f.columns if col.startswith("kelas")])

X, X_test = feats(train), feats(test)
assert list(X.columns) == list(X_test.columns)
print("feature matrix:", X.shape, "| seeds:", SEEDS)

# quartile_bin: rank the latent score, cut into 4 equal bins -> balanced labels
def show_quartile_bin():
    demo = np.array([0.1, 5.0, 2.0, 9.0, 3.0, 7.0, 1.0, 8.0])
    return quartile_bin(demo)
print("quartile_bin demo (ranks -> 0..3):", show_quartile_bin())


feature matrix: (3200, 52) | seeds: (42, 43, 44, 45, 46)
quartile_bin demo (ranks -> 0..3): [0 2 1 3 1 2 0 3]


## CRISP-DM 4 — Modeling

### The core idea: estimate a latent *rank*, then quartile-bin

Every candidate produces one continuous score per student; we z-normalise it,
quartile-bin the out-of-fold scores, and score accuracy. All estimators are fit
**inside** each CV fold (OOF) so nothing leaks. Because the classes are balanced and
we quartile-bin, **accuracy is purely a function of how well the score ranks students**.

### Ordinal decomposition — a strong, *independent* ranker

Instead of predicting the class directly, we decompose the ordered target into
three **binary** questions, each answered by its own XGBoost model:

$$S \;=\; P(y>0) \;+\; P(y>1) \;+\; P(y>2) \;=\; \sum_{k=0}^{2} P(y>k)$$

`S` lies in `[0, 3]` — the expected number of thresholds the student clears — a
natural latent *rank* that encodes the class ordering directly. Tuned binary params
(`n=550, d=4, lr=0.04`) come from the multi-seed grid in `scripts/exp_ordinal.py`.
It scores **0.5276**, comfortably above the *untuned* reg/clf pair (0.5158) — so
ordinality helps — but, crucially, **below the tuned v3 core (0.5334)**.


In [5]:
ORD = dict(n_estimators=550, max_depth=4, learning_rate=0.04, subsample=0.9,
           colsample_bytree=0.9, min_child_weight=3, reg_lambda=1.5)

def ordinal_oof_score(X, y, seed):
    """Out-of-fold cumulative ordinal score S = sum_k P(y>k). CV-safe."""
    cv = StratifiedKFold(5, shuffle=True, random_state=seed)
    s = np.zeros(len(y))
    for tr_i, va_i in cv.split(X, y):
        Xtr, Xva, ytr = X.iloc[tr_i], X.iloc[va_i], y.iloc[tr_i]
        for k in range(3):                      # binary: is target > k ?
            b = XGBClassifier(**ORD, random_state=seed, n_jobs=-1,
                              eval_metric="logloss", verbosity=0)
            b.fit(Xtr, (ytr > k).astype(int))
            s[va_i] += b.predict_proba(Xva)[:, 1]
    return s

# Quick in-notebook demo on 2 seeds (full 5-seed numbers are in the table below).
DEMO_SEEDS = (42, 43)
demo = [accuracy_score(y, quartile_bin(ordinal_oof_score(X, y, s))) for s in DEMO_SEEDS]
print("ordinal decomposition, demo seeds", DEMO_SEEDS, "->", [round(a, 4) for a in demo])
print("mean demo accuracy:", round(np.mean(demo), 4))


ordinal decomposition, demo seeds (42, 43) -> [0.5284, 0.5266]
mean demo accuracy: 0.5275


### What we tried and what we learned (full sweeps, seeds 42–46)

Measured in `scripts/exp_sweep.py`, `exp_final.py`, `exp_ordinal_ensemble.py`,
`exp_blend_ord_v3.py`:

| Configuration | CV | std | Verdict |
|---|---|---|---|
| reg + clf pair (**untuned**) | 0.5158 | 0.0051 | weak baseline |
| ordinal Σ P(y>k), tuned | 0.5276 | 0.0030 | strong, independent ranker |
| ordinal + reg + clf (untuned, ordinal-heavy) | 0.5196–0.5226 | — | ❌ untuned reg/clf dilute it |
| equal-weight 5-estimator blend | 0.5176 | 0.0045 | ❌ weak rankers drag it down |
| ordinal (XGB) + ordinal (HistGB) rank-avg | 0.5239 | 0.0024 | ❌ HGB ordinal is weaker |
| learned near-quartile thresholds | 0.5234–0.5266 | — | ❌ no gain — quartile optimal for balanced classes |
| ordinal seed-bagged (×3) | 0.5264 | 0.0012 | same mean, half the variance |
| **v3 reg/clf core, Optuna-tuned** | **0.5334** | 0.0021 | ✅ **best single model (LB 0.57916)** |
| ordinal + v3-core rank-blend (40/60) | 0.5343 | 0.0032 | ~noise-level gain; diversification hedge |

**Lessons that held up:**
1. **Tuning mattered more than the ordinal reframing.** The single biggest CV lever
   was the Optuna-tuned reg/clf core (0.5158 → 0.5334), not the ordinal trick (→0.5276).
   My initial "ordinal wins" claim compared against the wrong (untuned) baseline.
2. **CV cannot discriminate at ~0.001.** With std ≈ 0.003 and a public split of a few
   hundred rows, a 0.5343-vs-0.5334 gap is ~1 row — the LB proved it (v3 > ordinal by
   ~1 row). Do not chase sub-0.002 CV gains; they are noise.
3. **The ordinal model's worth is diversity.** It disagrees with v3 on ~15% of rows
   (symmetric ±1 boundary flips). Rank-blending the two is a legitimate private-score
   hedge — but only a hedge, not a confident improvement.
4. **Learned thresholds / weak-ranker blends don't help** — quartile cuts are optimal
   for balanced classes, and diluting a strong ranker with weak ones lowers CV.


## CRISP-DM 5 — Evaluation

We validate with **repeated 5-fold Stratified CV across seeds 42–46** (never a single
seed — a lone 0.502 seed earlier in the project was pure noise).

**Ranking of candidates (this harness):** v3 tuned core **0.5334** > ordinal **0.5276**
> untuned pair 0.5158. The ordinal + v3 rank-blend reaches ~0.534, statistically tied
with v3 alone.

**The decisive evidence is the leaderboard, not CV.** The ordinal candidate scored
**0.57500** public vs v3's **0.57916** — worse, confirming v3's CV edge is real even
though it is tiny. This is the project's recurring lesson in action: **at this problem's
resolution, CV differences below ~0.002 do not reliably predict the LB.** The confusion
matrix below shows why the models are so close — errors are almost entirely adjacent-class
(off by one), so any monotone ranker lands within a rows-wide band of the others.


In [6]:
# Confusion of the OOF ordinal predictions (seed 42) — where do errors concentrate?
oof = quartile_bin(ordinal_oof_score(X, y, 42))
cm = pd.crosstab(pd.Series(y, name="true"), pd.Series(oof, name="pred"))
print(cm)
adj = (np.abs(oof - y.values) <= 1).mean()
print(f"\nexact acc (seed 42): {(oof == y.values).mean():.4f}"
      f"   within-1-class: {adj:.4f}")
print("Errors are overwhelmingly adjacent-class (off by one) — consistent with an "
      "ordered latent score whose hardest calls are at the quartile boundaries.")


pred    0    1    2    3
true                    
0     532  213   55   13
1     205  310  212   69
2      56  213  323  192
3       7   64  210  526

exact acc (seed 42): 0.5284   within-1-class: 0.9175
Errors are overwhelmingly adjacent-class (off by one) — consistent with an ordered latent score whose hardest calls are at the quartile boundaries.


## CRISP-DM 6 — Deployment

Two candidates are packaged (both write balanced 200/200/200/200 submissions and run
the standard column/shape/id-order/label asserts). **No Kaggle submission is made.**

1. **`scripts/run_production_ordinal.py`** → `outputs/submission_ordinal.csv`
   — the seed-bagged ordinal model (CV 0.5264 ± 0.0012). Submitted: **LB 0.57500**.
   Regenerated by the cell below.
2. **`scripts/exp_blend_ord_v3.py`** → `outputs/submission_blend_ord_v3.csv`
   — the ordinal + v3-core rank-blend (CV ~0.534). Differs from v3 on ~15% of rows;
   a **diversification candidate** for private-score robustness, not a confident LB gain.

**Recommendation.** `outputs/submission_production_v3.csv` remains the best submission
(LB 0.57916). If you want to spend another submission on a *diversified* bet — one that
could help the private split even though CV can't prove it — the ord+v3 blend is the
principled choice. Otherwise, keep v3.


In [7]:
# Regenerate the production candidate (CV + submission CSV). ~a few minutes.
import subprocess, sys
res = subprocess.run([sys.executable, "scripts/run_production_ordinal.py"],
                     capture_output=True, text=True)
print(res.stdout[-800:])
print(pd.read_csv("outputs/submission_ordinal.csv").head())


nfeat=52 bag=3 params={'n_estimators': 550, 'max_depth': 4, 'learning_rate': 0.04, 'subsample': 0.9, 'colsample_bytree': 0.9, 'min_child_weight': 3, 'reg_lambda': 1.5}
  seed 42: 0.5284
  seed 43: 0.5269
  seed 44: 0.5259
  seed 45: 0.5262
  seed 46: 0.5247
CV 0.5264 +/- 0.0012
counts: {'0': 200, '1': 200, '2': 200, '3': 200}
wrote outputs/submission_ordinal.csv + metrics_ordinal.json

   id  target
0   3       2
1  12       2
2  14       2
3  18       3
4  28       0


### Summary

* **Problem framing:** "estimate an ordinal latent rank, then quartile-bin." Correct and
  useful — but the reframing alone was *not* the biggest lever.
* **Honest ranking (same CV harness):** v3 Optuna-tuned core **0.5334** > ordinal
  **0.5276** > untuned pair 0.5158. The LB agreed: v3 0.57916 > ordinal 0.57500. My first
  draft's "ordinal wins +1.1 pts" compared to the wrong baseline and is retracted here.
* **The real lesson:** for this dataset, **CV cannot discriminate below ~0.002** — the
  public split is only a few hundred rows and errors are all adjacent-class. Tuning and
  diversity matter; sub-noise CV chasing does not.
* **Data insight that still holds:** low univariate correlation ≠ no signal — the
  daily-activity and behavioural blocks are strong *multivariately*; only `kelas` (a
  near-unique ID) is dropped. Train/test PSI ≈ 0.
* **Deliverables:** `scripts/cv_harness.py` (features + CV), `run_production_ordinal.py`
  (ordinal candidate), `exp_blend_ord_v3.py` (diversified blend). Leakage-safe, guarded.

**Best action:** keep **v3** as the primary submission; use the **ord+v3 blend** only as a
deliberate diversification bet. Genuine next gains likely need *new signal* (feature
discovery, boundary specialists for the 1↔2 confusion), not more re-ranking of the same score.


---
# CRISP-DM — Iteration 2: Finding NEW signal (the real breakthrough)

The first iteration exhausted *re-ranking the same score* — every model plateaued
near 0.53 CV / 0.579 LB. Genuine gains needed **new signal**. This section finds it.

## Back to Data Understanding: hunt for signal orthogonal to the current model

Raw univariate correlation is blind to interactions. The right lens is **partial
correlation**: how much a candidate feature correlates with the target *after
removing what the current model already captures* (its OOF rank). A feature with
high partial correlation is genuinely new information.

Screening (in `scripts/exp_signal_screen.py`) surfaced one candidate far above the
rest — the **product of two individually-useless columns**:


In [8]:
import warnings, itertools, numpy as np, pandas as pd
from scipy.stats import spearmanr
warnings.filterwarnings("ignore")
train = pd.read_csv("kaggle/train.csv"); y = train["target"].values

scal = ["skor_motivasi","skor_kedisiplinan","skor_literasi","skor_minat_belajar",
        "indeks_kehadiran","skor_ekstrakurikuler","jarak_rumah_km","jumlah_saudara",
        "skor_tryout","urutan_ujian"]

print("Individually, the behavioural scores look like pure noise:")
for c in ["skor_motivasi","skor_kedisiplinan"]:
    print(f"  spearman({c}, target) = {spearmanr(train[c], y).correlation:+.3f}")

print("\nBut scan every pairwise PRODUCT — exactly one lights up:")
hits = []
for a, b in itertools.combinations(scal, 2):
    rho = spearmanr(train[a].values * train[b].values, y).correlation
    if abs(rho) > 0.05: hits.append((f"{a} * {b}", rho))
for n, r in sorted(hits, key=lambda t: -abs(t[1])):
    print(f"  {n:38s} spearman = {r:+.3f}")


Individually, the behavioural scores look like pure noise:
  spearman(skor_motivasi, target) = +0.013
  spearman(skor_kedisiplinan, target) = -0.001

But scan every pairwise PRODUCT — exactly one lights up:
  skor_motivasi * skor_kedisiplinan      spearman = +0.354


### Why a strong model never found this on its own

`skor_motivasi` alone ≈ 0.01. `skor_kedisiplinan` alone ≈ 0.00. Their **product ≈
0.354** — as strong as the best feature in the whole dataset, and it survives
*partial* correlation (0.33) against the model's existing rank, so it is almost
entirely new information.

This is the **greedy-tree blind spot**. XGBoost splits one feature at a time to
reduce loss. When *both* marginals are flat, no single split on either column helps,
so the tree never takes the first step toward the joint `motivasi × discipline`
region — even though both columns are sitting right there in the matrix. The
interaction is only discoverable if you **hand the model the product explicitly**.
(Confirmed: `mxd²` and `|mxd|` are ~0, so the signal is *signed and linear* in the
product — a clean planted interaction term. No second pairwise product exceeds 0.03.)


## Modeling + Evaluation: confirming the lift on the full harness

Adding the interaction (and a few daily-activity temporal-pattern features) to the
same 52-feature set, re-measured across seeds 42–46 (`scripts/exp_newsignal_confirm.py`
and `run_production_v4_interaction.py`):

| Model | base (52 feat) | + interaction | **+ all new signal** |
|---|---|---|---|
| Ordinal Σ P(y>k) | 0.5276 | 0.5732 | **0.5790 ± 0.0030** |
| v3 tuned core | 0.5334 | 0.5723 | 0.5771 ± 0.0020 |
| **Ordinal, seed-bagged (production v4)** | — | — | **0.5801 ± 0.0017** |

**+0.047 accuracy over the previous best (0.5334) — ~23× the 0.002 promotion bar,
with tight per-seed variance `[0.578, 0.583]`.**

Two independent reasons to trust it:
1. **No leakage** — `motivasi × discipline` is a target-free per-row transform,
   computed identically on train and test (train/test PSI = 0.02).
2. **The CV-vs-LB gap closes** — local CV (~0.580) now sits *inside* the public-LB
   band (~0.579), exactly what you expect when the missing structural signal is
   finally supplied. The earlier 0.53-CV/0.579-LB mismatch was this interaction.


In [9]:
# Regenerate the v4 candidate (interaction + ordinal, seed-bagged). ~a few minutes.
import subprocess, sys, pandas as pd
res = subprocess.run([sys.executable, "scripts/run_production_v4_interaction.py"],
                     capture_output=True, text=True)
print(res.stdout[-500:])
print(pd.read_csv("outputs/submission_v4_interaction.csv").head())


nfeat=56 bag=3
  seed 42: 0.5781
  seed 43: 0.5794
  seed 44: 0.5787
  seed 45: 0.5819
  seed 46: 0.5825
CV 0.5801 +/- 0.0017
counts: {0: 200, 1: 200, 2: 200, 3: 200}
wrote outputs/submission_v4_interaction.csv + metrics_v4_interaction.json

   id  target
0   3       0
1  12       2
2  14       1
3  18       3
4  28       0


## Deployment & recommendation

**`scripts/run_production_v4_interaction.py`** → `outputs/submission_v4_interaction.csv`
is the new best candidate: **CV 0.5801 ± 0.0017**, balanced 200/200/200/200,
leakage-safe, shift-stable, all submission asserts pass.

Ranking of everything built:

| Candidate | CV | LB |
|---|---|---|
| v3 tuned core (old best) | 0.5334 | 0.57916 |
| ordinal only | 0.5276 | 0.57500 |
| **v4 = ordinal + interaction signal** | **0.5801** | *untested — expected ≳ 0.579* |

**Recommendation:** v4 is a real, structural improvement (not CV noise), so it is the
candidate most likely to beat 0.57916 on the leaderboard. Submitting still requires
your explicit approval — **no Kaggle submission has been made.**

### What changed the game
The plateau was never a modelling problem — it was a **missing feature**. The dataset
hides its dominant signal in a single pairwise interaction (`skor_motivasi ×
skor_kedisiplinan`) that univariate correlation and greedy trees both overlook.
Finding it took the right *diagnostic* (partial correlation vs the current model),
not a stronger model. Remaining upside, if any, likely needs the same discipline:
search for structure the current score doesn't yet explain.


---
# CRISP-DM — Iteration 3: Temporal signal (push toward the leaderboard top)

Iteration 2's interaction feature confirmed the data is **synthetic with planted
structure** (LB jumped 0.579 → 0.600). If one structure was planted, others might
be. Chasing the #1 leaderboard score (0.6875) meant hunting for them systematically.

## Forensic data understanding: what kind of variables are these?

Per-column inspection revealed **8 of the "background" features are standardized
N(0,1) latent variables** (even `jarak_rumah_km` and `jumlah_saudara` run −3.5…+3.5).
That's the pool the interaction lived in. But scanning *all* pairwise / 3-way products
among them found **exactly one** (the motivasi×discipline term we already have) — that
vein is exhausted. The remaining signal had to be in the **sequence blocks**.


In [10]:
import warnings, numpy as np, pandas as pd
warnings.filterwarnings("ignore")
from scipy.stats import spearmanr
train = pd.read_csv("kaggle/train.csv"); y = train["target"].values
D = [f"aktivitas_hari_{i:02d}" for i in range(1, 17)]
W = [f"nilai_minggu_{i:02d}" for i in range(1, 13)]
Dm, Wm = train[D].to_numpy(float), train[W].to_numpy(float)

def acf_by_lag(M, maxlag):
    Mw = M - M.mean(1, keepdims=True); den = (Mw**2).sum(1) + 1e-9
    return {l: (Mw[:, :M.shape[1]-l]*Mw[:, l:]).sum(1)/den for l in range(1, maxlag+1)}

# The key discovery: per-student autocorrelation correlates strongly with target
dac, wac = acf_by_lag(Dm, 6), acf_by_lag(Wm, 6)
print("Per-student lag-autocorrelation vs target (spearman):")
print("  DAILY : " + "  ".join(f"lag{l}={spearmanr(dac[l], y).correlation:+.2f}" for l in range(1,7)))
print("  WEEKLY: " + "  ".join(f"lag{l}={spearmanr(wac[l], y).correlation:+.2f}" for l in range(1,7)))


Per-student lag-autocorrelation vs target (spearman):
  DAILY : lag1=-0.32  lag2=-0.19  lag3=+0.34  lag4=+0.25  lag5=-0.20  lag6=-0.18
  WEEKLY: lag1=+0.32  lag2=+0.28  lag3=+0.23  lag4=+0.06  lag5=-0.15  lag6=-0.30


## The generative structure: the target *controls the autocorrelation*

Averaging each block's autocorrelation function **within each class** exposes the
planted mechanism — the curves are cleanly **monotonic across the four classes**:


In [11]:
def acf_matrix(M, maxlag):
    Mw = M - M.mean(1, keepdims=True); den = (Mw**2).sum(1, keepdims=True) + 1e-9
    return np.column_stack([(Mw[:, :M.shape[1]-l]*Mw[:, l:]).sum(1) for l in range(1, maxlag+1)]) / den

for label, M in [("WEEKLY (persistence rises with class)", Wm), ("DAILY (period-~3 oscillation)", Dm)]:
    A = acf_matrix(M, 6)
    print(label)
    for c in range(4):
        print(f"  class {c}: " + "  ".join(f"{x:+.3f}" for x in A[y == c].mean(0)))


WEEKLY (persistence rises with class)
  class 0: +0.250  +0.153  +0.059  -0.040  -0.136  -0.232
  class 1: +0.321  +0.205  +0.079  -0.030  -0.145  -0.264
  class 2: +0.417  +0.269  +0.115  -0.028  -0.174  -0.314
  class 3: +0.504  +0.330  +0.158  -0.015  -0.196  -0.362
DAILY (period-~3 oscillation)
  class 0: +0.154  -0.339  -0.307  -0.047  +0.067  +0.072
  class 1: +0.102  -0.385  -0.222  +0.053  +0.045  +0.006
  class 2: +0.050  -0.423  -0.134  +0.128  -0.000  -0.032
  class 3: -0.030  -0.443  +0.010  +0.181  -0.101  -0.064


**Read the weekly table:** lag-1 autocorrelation climbs **0.25 → 0.32 → 0.42 →
0.50** from class 0 to 3 — higher performers have *more persistent* week-to-week grade
trajectories. **Daily** shows a period-~3 oscillation whose amplitude grows with class.
This is a second planted structure, entirely missed by mean/std/slope summaries.

### Why it was invisible before
- **Univariate:** each daily/weekly column alone ≈ 0 correlation.
- **Raw products:** daily values sit around 50, so raw products are swamped by the
  constant — only after **centering each column** do the lag-3 daily pairs light up.
- **Generic summaries** (mean/std/slope) don't encode *lag structure* at all.

The fix: hand the model **per-student lag-autocorrelation features** for both blocks.


## Modeling + Evaluation: the temporal lift

Adding lag-autocorrelation features (ordinal model, seed-bagged, seeds 42–46;
`scripts/exp_daily_lag.py`, `exp_v6_consolidate.py`, `run_production_v6_temporal.py`):

| Feature set | CV | std |
|---|---|---|
| v4 (interaction, no temporal) | 0.5793 | 0.0033 |
| + daily lag-autocorr | 0.6141 | 0.0034 |
| + daily **and** weekly lag-autocorr (**v6**) | **0.6171** | **0.0035** |

Ablations that shaped v6: **FFT power bins were redundant** with autocorr (added only
variance — dropped); **higher tree depth (5–6) overfit** — depth 4 stays best. A final
residual screen against the v6 rank found **nothing left above partial-corr 0.03** — both
planted structures are now captured; the remaining ceiling is per-student estimation
noise from short (12–16 point) series.


In [12]:
import json
m = json.load(open("outputs/metrics_v6_temporal.json"))
print("v6 (ordinal + interaction + temporal autocorrelation), seed-bagged")
print(f"  features        : {m['feature_shape'][1]}")
print(f"  CV accuracy     : {m['cv_mean_accuracy']:.4f} +/- {m['cv_std']:.4f}")
print(f"  per seed        : {m['per_seed']}")
print(f"  prediction dist : {m['prediction_counts']}")
# To fully regenerate from scratch (~a few min), uncomment:
# import subprocess, sys; print(subprocess.run([sys.executable, 'scripts/run_production_v6_temporal.py'], capture_output=True, text=True).stdout[-400:])


v6 (ordinal + interaction + temporal autocorrelation), seed-bagged
  features        : 51
  CV accuracy     : 0.6171 +/- 0.0035
  per seed        : [0.6184, 0.6125, 0.6166, 0.6228, 0.615]
  prediction dist : {'0': 200, '1': 200, '2': 200, '3': 200}


## Deployment & recommendation

**Progression across iterations (all seed-bagged ordinal, seeds 42–46):**

| Version | Key addition | CV | LB |
|---|---|---|---|
| v3 | tuned reg/clf core | 0.5334 | 0.57916 |
| **v4** | + `motivasi×discipline` interaction | 0.5801 | **0.6000** |
| **v6** | + daily/weekly lag-autocorrelation | **0.6171** | *untested* |

`scripts/run_production_v6_temporal.py` → **`outputs/submission_v6_temporal.csv`** is the
new best candidate: CV 0.6171 ± 0.0035, balanced 200/200/200/200, leakage-safe,
train/test-stable, all asserts pass. Given v4's CV 0.58 → LB 0.60 relationship, v6's
+0.037 CV should translate to a substantial LB gain.

**Recommendation:** submit `submission_v6_temporal.csv`. It is the candidate most likely
to approach/beat the 0.6875 leader. **No Kaggle submission has been made** — awaiting
your approval.

### The through-line
Every real gain came from **finding planted signal**, not from bigger models:
a hidden interaction (iter 2) and target-driven temporal autocorrelation (iter 3). The
enabling tool each time was the right *diagnostic* — partial correlation against the
current model's rank, and mean-structure-by-class — which sees what univariate
correlation and greedy trees cannot.
